## Суммаризация
1. Вид семантичской свертки, другие виды
2. Виды суммаризации. 
3. Подходы.(TextRank TF-IDF + ранжирование предложений, Глуюокие модели)
4. Связность

In [ ]:
!pip install --upgrade pip
!pip install sumy
!pip install transformers sentencepiece

In [42]:
import os
import re

path = "IMS_extracted"
texts = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('_rus.txt') and 'Abstract' not in file and 'KW' not in file:
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    text = f.read()

                    name = file[:-4]  # Без ".txt"
                    
                    # Ищем год в имени файла
                    match = re.search(r'20\d{2}', file)
                    year = match.group(0) if match else 'Unknown'
                    if year not in {'2020', '2021', '2022', '2023', '2024'}:
                        continue

                    texts.append((name, text, year))
            except Exception as e:
                print(f"Ошибка при чтении файла {file_path}: {e}")


In [43]:
texts[:1]

[('Bakhvalov_IMS_2020_rus',
  'Разработка и реализация методов генерации правил \nдля автоматической проверки правописания\nП.Я. Бахвалов\nУниверситет ИТМО\nВведение\nАвтоматическая проверка правописания – это задача автоматического обнаружения и исправления грамматических, стилистических и орфографических ошибок в тексте. Актуальность этой задачи определяется тем, что людям свойственно делать ошибки правописания. Далеко не все хорошо знакомы с правилами грамматики языка или только приступают к их изучению. Помимо этого, существуют болезни, такие как дисграфия, при которых человеку особенно тяжело правильно писать. Задача осложняется за счет таких факторов как неформализованная грамматика, свободный порядок слов в предложении, зависимость слов от контекста, постоянные изменения в языке, диалекты, сленг, жаргон, омонимы и др.\nПодходы к задаче автоматической проверки правописания можно разделить на три типа: основанные на правилах, основанные на методах машинного обучения и гибридные. П

In [44]:
df = pd.DataFrame(texts, columns=['names', 'texts', 'years'])

df

,names,texts,years
0,Bakhvalov_IMS_2020_rus,Разработка и реализация методов генерации прав...,2020
1,Grebennikov_IMS_2020_rus,Корпус русского рассказа начала XX века. \nПри...,2020
2,Khokhlova_IMS_2020_rus,Методы машинного обучения применительно к зада...,2020
3,Korablinov_IMS_2020_rus,Подготовка набора данных для вопросно-ответног...,2020
4,Kuznetsova_IMS_2020_rus,О возможности использования корпуса NOW в курс...,2020
5,Mikoni_IMS_2020_rus,Три подхода к определению понятий на основе со...,2020
6,Rogov_IMS_2020_rus,Применение деревьев решений для анализа сильны...,2020
7,Sokolova_IMS_2020_rus,К вопросу о формировании набора отношений для ...,2020
8,Antopolskii_IMS_2021_rus,Лингвистические ресурсы:\nевропейский опыт и у...,2021
9,Boyarsky_IMS_2021_rus,Устойчивые словосочетания в роли предлогов\nК....,2021


## Sumy

In [23]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alena\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
summarizer = LexRankSummarizer() #алгоритм LexRank из библиотеки sumy, основанный на графах

In [45]:
abstracts = []

In [ ]:
for text in df.texts:
  doc = text
  parser = PlaintextParser.from_string(doc, Tokenizer('russian'))
  summary = summarizer(parser.document, 3) #количество предложений в итоговом резюме
  abstracts.append([str(sentence) for sentence in summary]) #список предложений преобразуется в строку

In [47]:
abstracts[:10]

[['Тем самым мы предполагаем, что в категории ошибок нет ни одной другой ошибки кроме как из данной категории.',
  'Процесс генерации правил выглядит следующим образом.',
  'Такие же графы были сгенерированы для всего набора данных и для троек, включающих ошибку из набора данных.'],
 ['Корпус русского рассказа начала XX века.',
  'С этой целью в СПбГУ реализуется проект по созданию Корпуса рассказов русских писателей, охватывающего произведения возможно большего числа литераторов, написанные на русском языке с 1900 по 1930 гг. и опубликованные в периодических изданиях или отдельными брошюрами (подробнее о принципах построения корпуса см. [1, 2]).',
  'Обсуждение В данной статье описываются результаты лингвостатистического анализа частотного словаря, поостренного для выборки объемом 100 рассказов из Корпуса русских рассказов 1900 – 1930-х гг. Исследование затрагивает только первый из рассматриваемых в Корпусе периодов, а именно 1900 – 1913 гг. 100 наиболее частотных знаменательных слов 

In [50]:
annotations = []

In [51]:
for abstr in enumerate(abstracts, start=1):
  print(f'Аннотация статьи {abstr[0]}:')
  annotation = str(' '.join(abstr[1]))
  annotations.append(annotation)
  print(annotation, '\n\n')

Аннотация статьи 1:
Тем самым мы предполагаем, что в категории ошибок нет ни одной другой ошибки кроме как из данной категории. Процесс генерации правил выглядит следующим образом. Такие же графы были сгенерированы для всего набора данных и для троек, включающих ошибку из набора данных. 


Аннотация статьи 2:
Корпус русского рассказа начала XX века. С этой целью в СПбГУ реализуется проект по созданию Корпуса рассказов русских писателей, охватывающего произведения возможно большего числа литераторов, написанные на русском языке с 1900 по 1930 гг. и опубликованные в периодических изданиях или отдельными брошюрами (подробнее о принципах построения корпуса см. [1, 2]). Обсуждение В данной статье описываются результаты лингвостатистического анализа частотного словаря, поостренного для выборки объемом 100 рассказов из Корпуса русских рассказов 1900 – 1930-х гг. Исследование затрагивает только первый из рассматриваемых в Корпусе периодов, а именно 1900 – 1913 гг. 100 наиболее частотных знамен

In [52]:
annots = pd.DataFrame(annotations, columns=['annotations_sumy'])

In [53]:
sumy = pd.concat([df, annots], join='outer', axis=1)

In [54]:
sumy

,names,texts,years,annotations_sumy
0,Bakhvalov_IMS_2020_rus,Разработка и реализация методов генерации прав...,2020,"Тем самым мы предполагаем, что в категории оши..."
1,Grebennikov_IMS_2020_rus,Корпус русского рассказа начала XX века. \nПри...,2020,Корпус русского рассказа начала XX века. С это...
2,Khokhlova_IMS_2020_rus,Методы машинного обучения применительно к зада...,2020,Обзор методов Традиционные методы извлечения л...
3,Korablinov_IMS_2020_rus,Подготовка набора данных для вопросно-ответног...,2020,Подготовка набора данных для вопросно-ответног...
4,Kuznetsova_IMS_2020_rus,О возможности использования корпуса NOW в курс...,2020,Для проверки частотности из off-list списка бы...
5,Mikoni_IMS_2020_rus,Три подхода к определению понятий на основе со...,2020,Переведя все поясняющие слова в именительный п...
6,Rogov_IMS_2020_rus,Применение деревьев решений для анализа сильны...,2020,Для решения задачи атрибуции текстов хорошо за...
7,Sokolova_IMS_2020_rus,К вопросу о формировании набора отношений для ...,2020,Обобщение типов похожих риторических отношений...
8,Antopolskii_IMS_2021_rus,Лингвистические ресурсы:\nевропейский опыт и у...,2021,Федеративный поиск. Инфраструктура знаний. При...
9,Boyarsky_IMS_2021_rus,Устойчивые словосочетания в роли предлогов\nК....,2021,"В общем случае слова, входящие в состав фразем..."


## T5

In [35]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from transformers import AutoModelForSeq2SeqLM, T5TokenizerFast

C:\Users\alena\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
import torch
from transformers import AutoTokenizer, AutoModelWithLMHead, T5ForConditionalGeneration

## RuT5-base-sum

In [ ]:
model_name = "IlyaGusev/rut5_base_sum_gazeta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name) #класс модели T5, предназначенный для задач генерации последовательностей, в том числе суммаризации.

C:\Users\alena\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alena\.cache\huggingface\hub\models--IlyaGusev--rut5_base_sum_gazeta. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [56]:
output_dir = "Summarization"

# Фильтруем только нужные года
years = ['2020', '2021', '2022', '2023', '2024']

# Создаём папки под каждый год
output_dir = "Summarization"
for year in years:
    year_folder = os.path.join(output_dir, f"{year}_annotations")
    os.makedirs(year_folder, exist_ok=True)

In [ ]:
annotations_t5 = []

for text in df.texts:
  article_text = text
  #tokenizer преобразует текст в числовые идентификаторы токенов (input_ids), которые подаются на вход модели.
  input_ids = tokenizer([article_text], max_length=600, add_special_tokens=True, padding="max_length", truncation=True, return_tensors="pt")["input_ids"] # возвращает список сгенерированных последовательностей для батча. 
  output_ids = model.generate(input_ids=input_ids, min_length=50, max_length=250, no_repeat_ngram_size=4)[0] #запрещает повторение последовательностей из 4 токенов
  summary = tokenizer.decode(output_ids, skip_special_tokens=True)
  print(f'Аннотация текста:\n {summary}\n\n')
  annotations_t5.append(str(''.join(summary)))

Аннотация текста:
 Автоматическая проверка правописания – это задача автоматического обнаружения и исправления грамматических ошибок в тексте. Эксперты считают, что эта задача является одной из самых сложных задач для людей, которым свойственно делать ошибки правописания.


Аннотация текста:
 В Санкт-Петербурге создан Корпус рассказов русских писателей, охватывающий произведения возможно большого числа литераторов, написанных на русском языке с 1900 по 1930 гг. и опубликованные в периодических изданиях или отдельных брошюрах.


Аннотация текста:
 Выделение атрибутивных и глагольных коллокаций при помощи моделей машинного обучения на русских коллекциях текстов большого объема стало одним из методов, применяемых при решении лингвистических задач в связи с появлением как больших текстовых данных, так и технических возможностей.


Аннотация текста:
 Введение набора данных для вопросно-ответного поиска по базе знаний. Основная задача – преобразование вопроса на естественном языке в конструк

In [58]:
annots_t5 = pd.DataFrame(annotations_t5, columns=['annotations_t5'])

In [59]:
full = pd.concat([sumy, annots_t5], join='outer', axis=1)

In [60]:
full

,names,texts,years,annotations_sumy,annotations_t5
0,Bakhvalov_IMS_2020_rus,Разработка и реализация методов генерации прав...,2020,"Тем самым мы предполагаем, что в категории оши...",Автоматическая проверка правописания – это зад...
1,Grebennikov_IMS_2020_rus,Корпус русского рассказа начала XX века. \nПри...,2020,Корпус русского рассказа начала XX века. С это...,В Санкт-Петербурге создан Корпус рассказов рус...
2,Khokhlova_IMS_2020_rus,Методы машинного обучения применительно к зада...,2020,Обзор методов Традиционные методы извлечения л...,Выделение атрибутивных и глагольных коллокаций...
3,Korablinov_IMS_2020_rus,Подготовка набора данных для вопросно-ответног...,2020,Подготовка набора данных для вопросно-ответног...,Введение набора данных для вопросно-ответного ...
4,Kuznetsova_IMS_2020_rus,О возможности использования корпуса NOW в курс...,2020,Для проверки частотности из off-list списка бы...,О возможности использования корпуса NOW в курс...
5,Mikoni_IMS_2020_rus,Три подхода к определению понятий на основе со...,2020,Переведя все поясняющие слова в именительный п...,Определение понятий на основе собственных свой...
6,Rogov_IMS_2020_rus,Применение деревьев решений для анализа сильны...,2020,Для решения задачи атрибуции текстов хорошо за...,В работе «Атрибуция произведений Ф. М. Достоев...
7,Sokolova_IMS_2020_rus,К вопросу о формировании набора отношений для ...,2020,Обобщение типов похожих риторических отношений...,Корпусы с дискурсивной разметкой текста станов...
8,Antopolskii_IMS_2021_rus,Лингвистические ресурсы:\nевропейский опыт и у...,2021,Федеративный поиск. Инфраструктура знаний. При...,Европейский опыт и уроки для России: лингвисти...
9,Boyarsky_IMS_2021_rus,Устойчивые словосочетания в роли предлогов\nК....,2021,"В общем случае слова, входящие в состав фразем...",На русском языке возникает проблема снятия омо...


In [61]:
full.to_csv('summarization.csv')

In [62]:
for index, row in full.iterrows():
  year = row['years']
  name = row['names']
  text = row['annotations_t5']
  try:
    with open(f'Summarization/{year}_annotations/{name}_annotation.txt', 'w') as writefile:
      writefile.write(text)
  except:
    pass